In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/marquis03/bean-leaf-lesions-classification")

In [ ]:
Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: 
Your Kaggle Key: 
Dataset URL: https://www.kaggle.com/datasets/marquis03/bean-leaf-lesions-classification
Downloading bean-leaf-lesions-classification.zip to ./bean-leaf-lesions-classification
100%|██████████| 155M/155M [00:00<00:00, 1.28GB/s]

In [ ]:
import pandas as pd
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU,

In [ ]:
data_df = pd.read_csv("/content/bean-leaf-lesions-classification/train.csv")
print(data_df.head())


In [ ]:


                            image:FILE  category
0   train/healthy/healthy_train.98.jpg         0
1  train/healthy/healthy_train.148.jpg         0
2  train/healthy/healthy_train.306.jpg         0
3  train/healthy/healthy_train.305.jpg         0
4   train/healthy/healthy_train.40.jpg         0

In [ ]:
catagory = data_df['category'].unique()
print(catagory)
[0 1 2]

In [ ]:
print(data_df.shape[0])
1034

In [ ]:
train_dataset = data_df.sample(frac=0.8, random_state=7)
test_dataset = data_df.drop(train_dataset.index)
print(train_dataset.shape[0])
print(test_dataset.shape[0])


827
207

In [ ]:
transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float)
])

In [ ]:
Custom Dataset Class

In [ ]:
class CustomImageDataset(Dataset):
  def __init__(self, dataframe, transform=None):
    self.dataframe = dataframe
    self.transform = transform
    self.labels = torch.tensor(dataframe['category'].values).to(device)


  def __len__(self):
    return self.dataframe.shape[0]

  def __getitem__(self, idx):
    image_path = self.dataframe.iloc[idx, 0]
    labels = self.labels[idx]
    image = Image.open(image_path).convert('RGB')
    if self.transform:
      image = (self.transform(image)/255).to(device)
    return image, labels

In [ ]:
Create Dataset Objects

In [ ]:
train_dataset =CustomImageDataset(dataframe=train_dataset, transform = None)
test_dataset = CustomImageDataset(dataframe=test_dataset, transform = None)

In [ ]:

train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=True)